# Task 2 — Personalized Restaurant Recommendation Engine
**Intern:** Abhishek | **Enrollment:** CTI/A1/C358755
**Organization:** Cognifyz Technologies | **Domain:** Machine Learning

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## 2. Load & Clean Dataset

In [ ]:
df = pd.read_csv("../Dataset/Dataset.csv")
df["Cuisines"] = df["Cuisines"].fillna("Unknown")
df = df[df["Aggregate rating"] > 0].copy()
df["City_clean"] = df["City"].str.strip().str.lower()
df["Cuisines_clean"] = df["Cuisines"].str.lower()
df["Primary Cuisine"] = df["Cuisines"].apply(lambda x: x.split(",")[0].strip())
print(f"Working dataset: {df.shape[0]} rated restaurants, {df['City'].nunique()} cities")

## 3. Dataset Analysis

In [ ]:
# Top cuisines
top_cuisines = df["Primary Cuisine"].value_counts().head(15)
top_cuisines.sort_values().plot(kind="barh", figsize=(10,6), color="coral", edgecolor="white")
plt.title("Top 15 Primary Cuisines", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Price range distribution
df["Price range"].value_counts().sort_index().plot(
    kind="bar", figsize=(7,5), color=["#4CAF50","#2196F3","#FF9800","#F44336"],
    edgecolor="white", rot=0)
plt.title("Price Range Distribution (1=Cheap to 4=Premium)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Recommendation Engine

In [ ]:
def compute_score(row, cuisine_pref, budget_tier, min_rating, city_pref, weights):
    cuisine_score = 1.0 if cuisine_pref.lower() in row["Cuisines_clean"] else 0.0
    diff = abs(row["Price range"] - budget_tier)
    budget_score = 1.0 if diff == 0 else (0.5 if diff == 1 else 0.0)
    if row["Aggregate rating"] < min_rating:
        return -1
    rating_score = row["Aggregate rating"] / 4.9
    city_score = 1.0 if city_pref.lower() in row["City_clean"] else 0.0
    return (weights["cuisine"] * cuisine_score + weights["budget"] * budget_score +
            weights["rating"] * rating_score + weights["city"] * city_score)

def recommend(cuisine_pref="North Indian", budget_tier=2, min_rating=3.5,
              city_pref="New Delhi", top_n=10, weights=None):
    if weights is None:
        weights = {"cuisine": 0.35, "budget": 0.20, "rating": 0.30, "city": 0.15}
    scores = df.apply(lambda row: compute_score(row, cuisine_pref, budget_tier, min_rating, city_pref, weights), axis=1)
    mask = scores >= 0
    scored = df[mask].copy()
    scored["match_score"] = scores[mask].values
    if scored.empty:
        print("No matches found. Try relaxing filters.")
        return pd.DataFrame()
    cols = ["Restaurant Name","City","Cuisines","Average Cost for two","Price range","Aggregate rating","Votes","match_score"]
    return scored.sort_values("match_score", ascending=False).head(top_n)[cols].reset_index(drop=True)

print("Recommendation engine ready.")

## 5. Sample Recommendations

In [ ]:
print("SCENARIO 1: North Indian | Tier 2 | New Delhi | Rating ≥ 3.5")
s1 = recommend("North Indian", 2, 3.5, "New Delhi", 5)
s1

In [ ]:
print("SCENARIO 2: Italian | Tier 4 | Any City | Rating ≥ 4.0")
s2 = recommend("Italian", 4, 4.0, "", 5, {"cuisine":0.50,"budget":0.20,"rating":0.30,"city":0.0})
s2

In [ ]:
print("SCENARIO 3: Chinese | Tier 1 | Mumbai | Rating ≥ 3.0")
s3 = recommend("Chinese", 1, 3.0, "Mumbai", 5)
s3

## 6. Conclusions

**Approach:** Weighted content-based filtering using 4 criteria  
**Weights:** Cuisine (35%), Rating (30%), Budget (20%), City (15%)

**Key Findings:**
- North Indian is the most available cuisine for recommendation
- Higher price tiers correlate with higher ratings
- The system correctly surfaces relevant, highly-rated restaurants
- Weights can be tuned by the user for different use cases